# CondAptNet — Stage 1 Training (Colab / T4)

**Before running:** Set runtime to GPU (Runtime → Change runtime type → T4 GPU).  
**Steps:** GPU check → clone repo → download data → resume check → train → evaluate.

In [1]:
# Cell 1 — GPU check
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → GPU (T4)."
)

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024 ** 3
print(f"GPU  : {gpu.name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
assert vram_gb >= 12, f"Expected ≥12 GB VRAM, got {vram_gb:.1f} GB"
print("GPU check passed.")

GPU  : Tesla T4
VRAM : 14.6 GB
CUDA : 12.8
GPU check passed.


In [2]:
# Cell 2 — Clone repo and install dependencies
import subprocess, sys

result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/shivanshb828/CondAptNet.git"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Repo cloned.")
else:
    # Already cloned — pull latest
    subprocess.run(["git", "-C", "CondAptNet", "pull", "--quiet"],
                   check=True)
    print("Repo already present — pulled latest.")

import os
os.chdir("CondAptNet")
print(f"Working directory: {os.getcwd()}")

packages = [
    "fair-esm",
    "ViennaRNA",
    "scikit-learn",
    "scipy",
    "biopython",
    "gdown",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    check=True
)
print("Dependencies installed.")

Repo cloned.
Working directory: /content/CondAptNet
Dependencies installed.


In [3]:
# Cell 3 — Download data from Google Drive
# *** UPDATE DRIVE IDs before running ***
# Upload these files to your Google Drive and paste their IDs below:
#   data/processed/master_dataset_cleaned.csv  → MASTER_DATASET_ID
#   data/processed/vienna_cache.pkl           → VIENNA_CACHE_ID  (if updated)
#   data/processed/protein_embeddings/ folder → PROTEIN_EMBEDDINGS_ID
# train.py recomputes any missing embeddings on first run (~5-10 min on T4),
# so PROTEIN_EMBEDDINGS_ID="UPDATE_ME" is safe to leave if not uploaded yet.

import os
import gdown

MASTER_DATASET_ID     = "UPDATE_ME"  # upload master_dataset_cleaned.csv to Drive
VIENNA_CACHE_ID       = "1mRaGNcVsaSb7QywORqCHSIMQWcHXpvV2"  # update if pkl changed
PROTEIN_EMBEDDINGS_ID = "UPDATE_ME"  # upload protein_embeddings/ folder to Drive

os.makedirs("data/processed/protein_embeddings", exist_ok=True)
os.makedirs("models/checkpoints/pretrain",       exist_ok=True)

print("Downloading master_dataset_cleaned.csv ...")
gdown.download(
    f"https://drive.google.com/uc?id={MASTER_DATASET_ID}",
    "data/processed/master_dataset_cleaned.csv",
    quiet=False,
)

print("Downloading vienna_cache.pkl ...")
gdown.download(
    f"https://drive.google.com/uc?id={VIENNA_CACHE_ID}",
    "data/processed/vienna_cache.pkl",
    quiet=False,
)

if PROTEIN_EMBEDDINGS_ID != "UPDATE_ME":
    print("Downloading protein_embeddings folder ...")
    gdown.download_folder(
        f"https://drive.google.com/drive/folders/{PROTEIN_EMBEDDINGS_ID}",
        output="data/processed/protein_embeddings",
        quiet=False,
        use_cookies=False,
    )
else:
    print("Skipping protein_embeddings download — will compute on first train run (~5-10 min).")

# Verify
import pandas as pd
df = pd.read_csv("data/processed/master_dataset_cleaned.csv")
ready = (
    df["aptamer_sequence"].notna() &
    df["protein_sequence"].notna() &
    df["split"].isin(["train", "val", "test"])
).sum()
n_emb = len(os.listdir("data/processed/protein_embeddings"))
print(f"master_dataset_cleaned.csv : {len(df):,} rows  ({ready:,} training-ready)")
print(f"protein_embeddings cached  : {n_emb} .npy files")
print("Data download complete.")


In [4]:
# Cell 4 — Resume logic
# Finds the latest epoch checkpoint so training can continue after a disconnect.

import glob

CHECKPOINT_DIR = "models/checkpoints/pretrain"

existing = sorted(glob.glob(f"{CHECKPOINT_DIR}/epoch_*.pt"))

if existing:
    latest = existing[-1]
    import torch
    meta = torch.load(latest, map_location="cpu")
    last_epoch    = meta.get("epoch", "?")
    best_val_mcc  = meta.get("best_val_mcc", meta.get("val_mcc", float("nan")))
    patience      = meta.get("patience_count", "?")
    RESUME_FLAG   = "--resume"
    print(f"Checkpoint found  : {latest}")
    print(f"Last epoch        : {last_epoch}")
    print(f"Best val MCC      : {best_val_mcc:.4f}")
    print(f"Patience count    : {patience}")
    print("Will resume training from next epoch.")
else:
    RESUME_FLAG = ""
    print("No checkpoint found — starting fresh.")

No checkpoint found — starting fresh.


In [5]:
# Cell 5 — Train (T4 settings: batch_size=16, max_prot_len=128)
# batch_size=16 + max_prot_len=128 keeps peak VRAM under ~12 GB on T4.
# Increase batch_size to 32 only if VRAM headroom allows.
import subprocess, os

cmd = [
    "python", "scripts/training/train.py",
    "--max-epochs",    "100",
    "--batch-size",    "16",
    "--max-prot-len",  "128",
    "--checkpoint-dir", CHECKPOINT_DIR,
]
if RESUME_FLAG:
    cmd.append(RESUME_FLAG)

env = os.environ.copy()
env["PYTHONUNBUFFERED"]      = "1"
env["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"Training finished (exit code {proc.returncode}).")


In [ ]:
# Cell 6 — Evaluate best checkpoint
import subprocess, os

best_ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")

if not os.path.exists(best_ckpt):
    print(f"No best checkpoint at {best_ckpt} — run Cell 5 first.")
else:
    for split in ("val", "test"):
        print(f"\n{'='*55}")
        print(f"Evaluating on {split.upper()} split")
        print(f"{'='*55}")
        result = subprocess.run(
            [
                "python", "scripts/evaluation/evaluate.py",
                "--checkpoint", best_ckpt,
                "--split",      split,
            ],
            capture_output=False, text=True,
        )